In [21]:
import re
import sqlite3
import logging
import pandas as pd
from pathlib import Path

In [22]:
DB_PATH          = "railway.db"
BLOCK_TRIGGER    = "EXCEPTION REPORT"
HEADER_SCAN_ROWS = 12   # rows to scan for metadata at top of each block
MIN_TABLE_COLS   = 2    # minimum populated cells in a row to be a table header
LOG_FORMAT       = "%(asctime)s [%(levelname)s] %(message)s"

In [23]:
logging.basicConfig(level=logging.INFO, format=LOG_FORMAT)
log = logging.getLogger(__name__)

In [24]:
DATASET_TYPE_MAP = [
    # ── filename keywords (checked first) ──────────────────────────────
    ("sod exception",           "sod_data"),
    ("lip flow",                "lip_flow_data"),
    ("lip_flow",                "lip_flow_data"),
    ("vertical wear",           "vertical_wear_data"),
    ("vertical_wear",           "vertical_wear_data"),
    ("lateral rail wear",       "lateral_wear_data"),
    ("lateral wear",            "lateral_wear_data"),
    ("lateral_wear",            "lateral_wear_data"),
    ("sleeper defect",          "sleeper_defects_data"),
    ("sleeper_defect",          "sleeper_defects_data"),
    ("sleeper",                 "sleeper_defects_data"),
    ("rail defect",             "rail_defects_data"),
    ("rail_defect",             "rail_defects_data"),
    ("fittings",                "fittings_data"),
    ("fitting",                 "fittings_data"),
    ("ballast and vegetation",  "ballast_vegetation_data"),
    ("ballast & vegetation",    "ballast_vegetation_data"),
    ("ballast",                 "ballast_data"),
    ("vegetation",              "vegetation_data"),
    # ── header / defect field keywords ─────────────────────────────────
    ("sod",                     "sod_data"),
    ("geometry",                "geometry_data"),
    ("gauge",                   "gauge_data"),
    ("squat",                   "squat_data"),
    ("corrugation",             "corrugation_data"),
    ("head check",              "head_check_data"),
]

In [25]:
def _is_numeric(val) -> bool:
    try:
        float(str(val).strip())
        return True
    except (ValueError, TypeError):
        return False

In [26]:
def detect_blocks(df: pd.DataFrame) -> list:
    """
    Find every row containing EXCEPTION REPORT.
    Slice the sheet into one sub-DataFrame per block.
    Everything above the first trigger is ignored.
    """
    trigger_rows = []
    for idx, row in df.iterrows():
        row_text = " ".join(str(v) for v in row.values if pd.notna(v))
        if BLOCK_TRIGGER.lower() in row_text.lower():
            trigger_rows.append(idx)
 
    if not trigger_rows:
        return []
 
    blocks = []
    for i, start in enumerate(trigger_rows):
        end = trigger_rows[i + 1] if i + 1 < len(trigger_rows) else df.index[-1] + 1
        block = df.loc[start:end - 1].reset_index(drop=True)
        blocks.append(block)
 
    log.info(f"  Detected {len(blocks)} block(s).")
    return blocks

In [27]:
def extract_metadata(block: pd.DataFrame) -> dict:
    """
    Flatten the first HEADER_SCAN_ROWS rows into a single string.
    Extract metadata fields with regex.
    """
    cells = []
    for _, row in block.iloc[:HEADER_SCAN_ROWS].iterrows():
        for val in row.values:
            if pd.notna(val) and str(val).strip():
                cells.append(str(val).strip())
 
    full_text = " | ".join(cells)
 
    def find(pattern, default=""):
        m = re.search(pattern, full_text, re.IGNORECASE)
        return m.group(1).strip() if m else default
 
    return {
        "full_header_text": full_text,
        "section":   find(r"section[:\s#-]*([A-Z0-9/ ]+?)(?:\||$|km|trc|run)"),
        "trc_no":    find(r"trc[\s#no.:]*([A-Z0-9\-/]+)"),
        "run_date":  find(r"(?:run\s*date|date)[:\s]*(\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4})"),
        "run_no":    find(r"run[\s#no.:]*(\d+)"),
        "defect":    find(r"defect[:\s]*([A-Za-z0-9 _\-]+?)(?:\||$|rail|section)"),
        "rail_side": find(r"\b(left|right|lh|rh)\b"),
        "km_range":  find(r"km[:\s]*([\d.]+\s*[-–to]+\s*[\d.]+)"),
    }

In [28]:
SERIAL_NO_PATTERN = re.compile(
    r"^\s*(?:s\.?\s*(?:r|l)?\.?\s*no\.?|s\s*no\.?|sno\.?|sl\.?\s*no\.?|sr\.?\s*no\.?|no\.?)\s*$",
    re.IGNORECASE
)

In [29]:
def _is_serial_no_row(row: pd.Series) -> bool:
    for val in row.values:
        if pd.notna(val) and SERIAL_NO_PATTERN.match(str(val).strip()):
            return True
    return False

In [30]:
def _count_populated(row: pd.Series) -> int:
    """Count non-null, non-empty cells in a row."""
    return sum(
        1 for v in row.values
        if pd.notna(v) and str(v).strip() not in ("", "nan")
    )

In [31]:
def _find_table_header_row(block: pd.DataFrame) -> int | None:
    """
    FIX 2: Two-stage header detection.
 
    Stage 1 (primary): Look for a row where any cell matches the S.No pattern.
    Stage 2 (fallback): After skipping the metadata header area, find the first
                        row with MIN_TABLE_COLS+ populated cells that is followed
                        by at least one row with numeric data in its first cell.
                        This handles files that have no S.No column at all.
    """
    # Stage 1 — S.No / Sr.No / Sl.No etc.
    for i, row in block.iterrows():
        if _is_serial_no_row(row):
            log.debug(f"    Header found via S.No pattern at row {i}")
            return i
 
    # Stage 2 — fallback: skip header area, find first populated row whose
    # next non-empty row starts with a numeric value
    search_start = min(HEADER_SCAN_ROWS, len(block) - 1)
    rows = list(block.iloc[search_start:].iterrows())
 
    for pos, (i, row) in enumerate(rows):
        if _count_populated(row) < MIN_TABLE_COLS:
            continue
        # Check whether the row immediately below contains real data
        next_rows = [r for _, r in rows[pos + 1:pos + 4]
                     if _count_populated(r) >= MIN_TABLE_COLS]
        if not next_rows:
            continue
        first_val = next(
            (v for v in next_rows[0].values
             if pd.notna(v) and str(v).strip() not in ("", "nan")),
            None
        )
        if first_val is not None:
            log.debug(f"    Header found via fallback at row {i}")
            return i
 
    return None

In [32]:
def _find_table_header_row(block: pd.DataFrame) -> int | None:
    """
    FIX 2: Two-stage header detection.
 
    Stage 1 (primary): Look for a row where any cell matches the S.No pattern.
    Stage 2 (fallback): After skipping the metadata header area, find the first
                        row with MIN_TABLE_COLS+ populated cells that is followed
                        by at least one row with numeric data in its first cell.
                        This handles files that have no S.No column at all.
    """
    # Stage 1 — S.No / Sr.No / Sl.No etc.
    for i, row in block.iterrows():
        if _is_serial_no_row(row):
            log.debug(f"    Header found via S.No pattern at row {i}")
            return i
 
    # Stage 2 — fallback: skip header area, find first populated row whose
    # next non-empty row starts with a numeric value
    search_start = min(HEADER_SCAN_ROWS, len(block) - 1)
    rows = list(block.iloc[search_start:].iterrows())
 
    for pos, (i, row) in enumerate(rows):
        if _count_populated(row) < MIN_TABLE_COLS:
            continue
        # Check whether the row immediately below contains real data
        next_rows = [r for _, r in rows[pos + 1:pos + 4]
                     if _count_populated(r) >= MIN_TABLE_COLS]
        if not next_rows:
            continue
        first_val = next(
            (v for v in next_rows[0].values
             if pd.notna(v) and str(v).strip() not in ("", "nan")),
            None
        )
        if first_val is not None:
            log.debug(f"    Header found via fallback at row {i}")
            return i
 
    return None

In [33]:
def extract_table(block: pd.DataFrame):
    """
    Locate the table header row (S.No or fallback).
    Build a DataFrame from rows below it.
    Drop footer/summary rows where the first cell is not numeric.
    """
    header_row_idx = _find_table_header_row(block)
 
    if header_row_idx is None:
        log.warning("  Table header row not found — skipping block.")
        return None
 
    header     = block.loc[header_row_idx].tolist()
    data_rows  = block.loc[header_row_idx + 1:].reset_index(drop=True)
 
    if data_rows.empty:
        log.warning("  No data rows found below header.")
        return None
 
    # Assign column names
    data_rows.columns = [
        str(h).strip() if pd.notna(h) and str(h).strip() else f"col_{i}"
        for i, h in enumerate(header)
    ]
 
    # Handle optional sub-header row (Km / Meter under Start / End Location)
    if not data_rows.empty:
        first_row = data_rows.iloc[0]
        sub_vals  = [str(v).strip().lower() for v in first_row.values if pd.notna(v)]
        if sub_vals and all(v in ("km", "meter", "m", "nan", "") for v in sub_vals):
            combined = []
            for col, sub in zip(data_rows.columns, first_row.values):
                sub_str = str(sub).strip()
                if pd.notna(sub) and sub_str.lower() not in ("nan", ""):
                    combined.append(f"{col}_{sub_str}")
                else:
                    combined.append(col)
            data_rows.columns = combined
            data_rows = data_rows.iloc[1:].reset_index(drop=True)
 
    # FIX 5: Drop footer / total / summary rows
    # Keep only rows where the first non-empty cell is numeric
    def first_nonempty(row):
        for v in row.values:
            if pd.notna(v) and str(v).strip() not in ("", "nan"):
                return v
        return None
 
    mask      = data_rows.apply(lambda r: _is_numeric(first_nonempty(r)), axis=1)
    data_rows = data_rows[mask].reset_index(drop=True)
 
    return data_rows if not data_rows.empty else None

In [34]:
def clean_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    - Drop all-NaN rows and auto-generated columns
    - Standardise names to snake_case
    - FIX 3: Deduplicate column names
    - FIX 5 (second pass): Remove any remaining non-numeric serial rows
    """
    df = df.dropna(how="all").reset_index(drop=True)
 
    # Remove auto-generated / unnamed columns
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed|^col_\d+$", na=False)]
 
    # snake_case column names
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^\w]", "_", regex=True)
        .str.replace(r"_+",   "_", regex=True)
        .str.strip("_")
    )
    df = df.loc[:, df.columns != ""]
 
    # FIX 3: Deduplicate — s_no → s_no, s_no_1, s_no_2 ...
    seen, new_cols = {}, []
    for col in df.columns:
        if col in seen:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            new_cols.append(col)
    df.columns = new_cols
 
    # FIX 5 second pass: if there is a recognised serial column, keep numeric rows only
    sno_col = next((c for c in df.columns if re.match(r"^s_?r?l?_?no", c)), None)
    if sno_col:
        df = df[df[sno_col].apply(_is_numeric)].reset_index(drop=True)
 
    return df

In [35]:
def detect_dataset_type(metadata: dict, file_name: str = "") -> str:
    """
    FIX 6: Check filename FIRST (most reliable), then header text.
    Returns the matching table name or 'general_data'.
    """
    # Build search text — filename first so it dominates
    text = (file_name + " " +
            metadata.get("full_header_text", "") + " " +
            metadata.get("defect", "")).lower()
 
    for keyword, table_name in DATASET_TYPE_MAP:
        if keyword in text:
            log.debug(f"  Dataset type → '{table_name}' (matched '{keyword}')")
            return table_name
 
    log.debug("  Dataset type → 'general_data' (no match)")
    return "general_data"

In [36]:
def _get_existing_columns(conn: sqlite3.Connection, table_name: str) -> list:
    cur = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name=?",
        (table_name,)
    )
    if cur.fetchone() is None:
        return []
    return [r[1] for r in conn.execute(f"PRAGMA table_info('{table_name}')")]

In [37]:
 def _add_missing_columns(conn: sqlite3.Connection, table_name: str, cols: list):
    existing = _get_existing_columns(conn, table_name)
    for col in cols:
        if col not in existing:
            try:
                conn.execute(f"ALTER TABLE '{table_name}' ADD COLUMN '{col}' TEXT")
                log.debug(f"  ALTER TABLE: added column '{col}' to '{table_name}'")
            except Exception as e:
                log.warning(f"  Could not add column '{col}': {e}")

In [38]:
def store_to_sql(df: pd.DataFrame, table_name: str, db_path: str = DB_PATH):
    """
    FIX 4: Schema-flexible append.
    Creates table on first write; ALTER TABLE for new columns on subsequent writes.
    """
    try:
        with sqlite3.connect(db_path) as conn:
            existing = _get_existing_columns(conn, table_name)
 
            if not existing:
                df.to_sql(table_name, conn, if_exists="append", index=False)
            else:
                _add_missing_columns(conn, table_name, df.columns.tolist())
                existing = _get_existing_columns(conn, table_name)
                common   = [c for c in df.columns if c in existing]
                if not common:
                    log.warning(f"  No common columns for '{table_name}' — skipping.")
                    return
                df[common].to_sql(table_name, conn, if_exists="append", index=False)
 
        log.info(f"  Stored {len(df)} row(s) → '{table_name}'")
 
    except Exception as e:
        log.error(f"  DB write error for '{table_name}': {e}")

In [39]:
def process_folder(folder_path: str, db_path: str = DB_PATH):
    """
    Walk every .xlsx in folder_path.
    For each file → sheet → EXCEPTION REPORT block:
      1. Extract metadata
      2. Extract & clean table  (drop footer rows)
      3. Attach metadata columns
      4. Detect dataset type from filename + header
      5. Write to SQLite (schema-flexible)
    """
    folder      = Path(folder_path)
    xlsx_files  = sorted(folder.glob("*.xlsx"))
 
    if not xlsx_files:
        log.warning(f"No .xlsx files found in: {folder_path}")
        return
 
    log.info(f"Found {len(xlsx_files)} Excel file(s) in '{folder_path}'.")
 
    # ── Summary counters ───────────────────────────────────────────────
    total_blocks  = 0
    total_rows    = 0
    tables_written: dict[str, int] = {}
 
    for file_path in xlsx_files:
        log.info(f"\n{'='*60}")
        log.info(f"FILE: {file_path.name}")
 
        try:
            xl = pd.ExcelFile(file_path, engine="openpyxl")
        except Exception as e:
            log.error(f"  Cannot open '{file_path.name}': {e}")
            continue
 
        for sheet_name in xl.sheet_names:
            log.info(f"  Sheet: '{sheet_name}'")
            try:
                raw_df = xl.parse(sheet_name, header=None, dtype=str)
            except Exception as e:
                log.error(f"  Cannot read '{sheet_name}': {e}")
                continue
 
            if raw_df.empty:
                continue
 
            blocks = detect_blocks(raw_df)
            if not blocks:
                continue
 
            for b_idx, block in enumerate(blocks):
                log.info(f"  Block {b_idx + 1}/{len(blocks)} ...")
                total_blocks += 1
                try:
                    metadata = extract_metadata(block)
                    table_df = extract_table(block)
 
                    if table_df is None or table_df.empty:
                        log.warning(f"    No usable table — skipping.")
                        continue
 
                    table_df = clean_table(table_df)
                    if table_df.empty:
                        log.warning(f"    Empty after cleaning — skipping.")
                        continue
 
                    # Attach metadata columns
                    table_df["section"]     = metadata.get("section",   "")
                    table_df["trc_no"]      = metadata.get("trc_no",    "")
                    table_df["run_date"]    = metadata.get("run_date",  "")
                    table_df["run_no"]      = metadata.get("run_no",    "")
                    table_df["defect"]      = metadata.get("defect",    "")
                    table_df["rail_side"]   = metadata.get("rail_side", "")
                    table_df["km_range"]    = metadata.get("km_range",  "")
                    table_df["source_file"] = file_path.name
                    table_df["sheet_name"]  = sheet_name
 
                    tbl = detect_dataset_type(metadata, file_path.name)
                    store_to_sql(table_df, tbl, db_path)
 
                    total_rows += len(table_df)
                    tables_written[tbl] = tables_written.get(tbl, 0) + len(table_df)
 
                except Exception as e:
                    log.error(f"    Block {b_idx + 1} failed: {e}", exc_info=True)
 
    # ── Final summary ──────────────────────────────────────────────────
    log.info(f"\n{'='*60}")
    log.info(f"PIPELINE COMPLETE")
    log.info(f"  Total blocks processed : {total_blocks}")
    log.info(f"  Total rows written     : {total_rows}")
    log.info(f"  Tables created/updated :")
    for tbl, rows in sorted(tables_written.items()):
        log.info(f"    {tbl:<35} {rows:>6} rows")
    log.info(f"  Database : {db_path}")
    log.info(f"{'='*60}")

In [40]:
# AFTER (hardcoded path)
if __name__ == "__main__":

    input_folder = r"D:\\ENGINEER\\IndianRailwaysProject\\data"   # ← paste your folder path here
    output_db    = "railway.db"                         # ← DB will be created here

    process_folder(input_folder, output_db)

2026-04-09 00:30:00,728 [INFO] Found 8 Excel file(s) in 'D:\\ENGINEER\\IndianRailwaysProject\\data'.
2026-04-09 00:30:00,730 [INFO] 
2026-04-09 00:30:00,730 [INFO] FILE: 1 SOD exception.xlsx
2026-04-09 00:30:01,089 [INFO]   Sheet: 'KYN-MMR UP'
2026-04-09 00:30:01,098 [INFO]   Sheet: 'KYN-MMR DN'
2026-04-09 00:30:01,100 [INFO]   Detected 1 block(s).
2026-04-09 00:30:01,100 [INFO]   Block 1/1 ...
2026-04-09 00:30:01,106 [WARNING]     No usable table — skipping.
2026-04-09 00:30:01,107 [INFO]   Sheet: 'MMR-CSN UP'
2026-04-09 00:30:01,108 [INFO]   Detected 1 block(s).
2026-04-09 00:30:01,109 [INFO]   Block 1/1 ...
2026-04-09 00:30:01,110 [WARNING]     No usable table — skipping.
2026-04-09 00:30:01,111 [INFO]   Sheet: 'MMR-CSN DN'
2026-04-09 00:30:01,114 [INFO]   Detected 1 block(s).
2026-04-09 00:30:01,114 [INFO]   Block 1/1 ...
2026-04-09 00:30:01,116 [WARNING]     No usable table — skipping.
2026-04-09 00:30:01,117 [INFO]   Sheet: 'MMR-CSN 3rd'
2026-04-09 00:30:01,164 [INFO]   Detected 